In [0]:
%run ../cralf_config/nb_00_config_lakeflow

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
orders_path = "/Volumes/customer_revenue_analytics/cralf_wh/raw/cralf_orders.json"

In [0]:
orders_schema = StructType([
    StructField("order_id", StringType()),
    StructField("user_id", StringType()),
    StructField("product_id", StringType()),
    StructField("quantity", IntegerType()),
    StructField("order_ts", StringType())
])

orders_df = (
    spark.read.format("json")
        .schema(orders_schema)
        .load(orders_path)
)

In [0]:
df_dedup = orders_df.dropDuplicates(["order_id"])

In [0]:
df_ts = (
    df_dedup
        .withColumn("order_ts", to_timestamp("order_ts"))
        .withColumn("order_date", to_date("order_ts"))
)

In [0]:
df_clean = (
    df_ts
        .filter(col('order_id').isNotNull())
        .filter(col('quantity') > 0)
)

In [0]:
orders_tbl_exists = spark.catalog.tableExists(orders_table)

In [0]:
if not orders_tbl_exists:
    df_clean.write.format("delta")\
        .partitionBy("order_date")\
        .saveAsTable(orders_table)
else:
    df_clean.write.format("delta")\
        .mode("append")\
        .saveAsTable(orders_table)

In [0]:
display(
    spark.sql(f"show partitions {orders_table}")
)